# 04 — Attention causale

Projet GPT reproductible · Notebook en français.


## 1. Research objective

Question : comment interdire la lecture des tokens futurs ? Hypothèse : masque supérieur sur les scores avant softmax, poids futurs nuls et sortie (B,T,D).


## 2. Configuration

Seed 2026; device auto; projections Q/K/V apprenables; scores divisés par √d_head; masque enregistré en buffer.


## 3. Implementation

CausalSelfAttention est une classe nn.Module de référence à une tête. Entrée/sortie (batch,séquence,d_model).


In [1]:
import math,torch
from torch import nn
SEED=2026; torch.manual_seed(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
class CausalSelfAttention(nn.Module):
 def __init__(self,d_model,d_head,context_length,dropout=0.,qkv_bias=False):
  super().__init__()
  if min(d_model,d_head,context_length)<=0 or not 0<=dropout<1: raise ValueError('Configuration invalide')
  self.d_model,self.d_head,self.context_length=d_model,d_head,context_length
  self.q=nn.Linear(d_model,d_head,bias=qkv_bias); self.k=nn.Linear(d_model,d_head,bias=qkv_bias); self.v=nn.Linear(d_model,d_head,bias=qkv_bias)
  self.proj=nn.Linear(d_head,d_model,bias=False); self.drop=nn.Dropout(dropout)
  self.register_buffer('mask',torch.triu(torch.ones(context_length,context_length,dtype=torch.bool),diagonal=1))
 def forward(self,x,return_attention=False):
  if x.ndim!=3 or x.shape[-1]!=self.d_model: raise ValueError('Entrée attendue (B,T,d_model)')
  b,t,d=x.shape
  if t>self.context_length: raise ValueError('Séquence au-delà du contexte')
  q,k,v=self.q(x),self.k(x),self.v(x)
  scores=(q@k.transpose(-2,-1))/math.sqrt(self.d_head)
  scores=scores.masked_fill(self.mask[:t,:t],torch.finfo(scores.dtype).min)
  w=self.drop(torch.softmax(scores,dim=-1)); out=self.proj(w@v)
  return (out,w) if return_attention else out
layer=CausalSelfAttention(32,16,12).to(DEVICE)
x=torch.randn(2,6,32,device=DEVICE); out,w=layer(x,True)
print('entrée',x.shape,'sortie',out.shape,DEVICE)


entrée torch.Size([2, 6, 32]) sortie torch.Size([2, 6, 32]) cuda


## 4. Unit tests

Tester les formes, somme des poids, masque et invariance aux futurs.


In [2]:
m=CausalSelfAttention(8,4,6).eval(); x=torch.randn(2,5,8); y,w=m(x,True)
assert y.shape==(2,5,8) and w.shape==(2,5,5)
assert torch.allclose(w.sum(-1),torch.ones(2,5),atol=1e-6)
assert torch.count_nonzero(torch.triu(w,diagonal=1))==0
x2=x.clone(); x2[:,4]+=100
with torch.no_grad(): assert torch.allclose(m(x)[:,:4],m(x2)[:,:4],atol=1e-6)
try: m(torch.randn(1,7,8))
except ValueError: pass
else: raise AssertionError('Contexte excessif accepté')
print('Tests attention causale / anti-fuite: OK')


Tests attention causale / anti-fuite: OK


## 5. Experiments

Comparer deux initialisations de même graine.


In [3]:
def build(seed):
 torch.manual_seed(seed); return CausalSelfAttention(8,4,6).eval()
p=torch.randn(1,4,8)
with torch.no_grad(): reproducible=torch.equal(build(SEED)(p),build(SEED)(p))
assert reproducible
{'input':tuple(x.shape),'output':tuple(out.shape),'future_weight_max':float(torch.triu(w,diagonal=1).max()),'row_sums':w.sum(-1)[0].tolist(),'same_seed':reproducible}


C:\Users\sail_\AppData\Local\Temp\ipykernel_17060\1030157652.py:6: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:837.)
  {'input':tuple(x.shape),'output':tuple(out.shape),'future_weight_max':float(torch.triu(w,diagonal=1).max()),'row_sums':w.sum(-1)[0].tolist(),'same_seed':reproducible}


{'input': (2, 5, 8),
 'output': (2, 6, 32),
 'future_weight_max': 0.0,
 'row_sums': [1.0, 1.0, 1.0, 0.9999999403953552, 1.0000001192092896],
 'same_seed': True}

## 6. Results

Formes, poids futurs maximaux et somme des lignes.


In [4]:
{"entrée": tuple(x.shape), "sortie": tuple(out.shape), "poids_futurs_max": float(torch.triu(w, diagonal=1).max()), "somme_poids": w.sum(-1)[0].tolist()}


{'entrée': (2, 5, 8),
 'sortie': (2, 6, 32),
 'poids_futurs_max': 0.0,
 'somme_poids': [1.0, 1.0, 1.0, 0.9999999403953552, 1.0000001192092896]}

## 7. Discussion

Sans padding mask, cache KV ni FlashAttention. `eval()` désactive dropout.
